# 02. 작은 단백질 MLM을 처음부터 학습하기 (합성 서열)

**등급**: 🟢 Mac 로컬 (CPU/MPS) · **예상 시간**: 2~4분 · **네트워크**: 불필요

**질문**: Masked language modeling만으로 모델이 서열 안의 **위치 간 의존성**(공진화와 비슷한 신호)을 학습하는가?

**설계**
- 길이 64의 합성 "단백질" 서열. 각 위치는 비균일한 잔기 빈도에서 독립 추출된다.
- 단, 12개 위치 쌍 (i, j)에는 **커플링**이 있다: 위치 j의 잔기가 위치 i의 잔기에 의해 (치환표로) 거의 결정된다. 10% 확률로 커플링이 깨진다(노이즈).
- 작은 Transformer 인코더(PyTorch)를 MLM으로 학습한 뒤, **커플링 위치를 가렸을 때 파트너를 보고 맞히는지**를 본다.

> 실제 단백질의 공진화/접촉과 같은 현상을 **단순화한 장난감**이다. 수치가 실제 PLM의 성능을 뜻하지 않는다.
> 이 아이디어의 실제 사례: Rao et al. 2021 (attention map이 contact를 학습함) — reading_list 02.

In [ ]:
import math
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
# 모델이 매우 작아서 CPU가 더 빠를 수도 있다. 필요하면 아래 줄의 주석을 풀어 강제로 CPU 사용:
# DEVICE = torch.device("cpu")
print("device:", DEVICE, "| torch", torch.__version__)

## 1. 합성 데이터

In [ ]:
AA = "ACDEFGHIKLMNPQRSTVWY"
V = len(AA)          # 잔기 종류 20
MASK_ID = V          # 마스크 토큰 id = 20
L = 64               # 서열 길이
N_PAIRS = 12
NOISE = 0.1          # 커플링이 깨질 확률

rs = np.random.RandomState(SEED)
background = rs.dirichlet(np.ones(V) * 2.0)          # 비균일한 잔기 빈도
perm_pos = rs.permutation(L)
PAIRS = [(int(perm_pos[2 * k]), int(perm_pos[2 * k + 1])) for k in range(N_PAIRS)]  # (i, j): i가 j를 결정
PERMS = [rs.permutation(V) for _ in range(N_PAIRS)]
coupled_positions = sorted({p for pair in PAIRS for p in pair})
print("coupled pairs (i->j):", PAIRS[:4], "...")
print("coupled positions:", len(coupled_positions), "of", L)


def sample(n, rs=rs):
    X = rs.choice(V, size=(n, L), p=background)
    for (i, j), pm in zip(PAIRS, PERMS):
        keep = rs.rand(n) > NOISE
        X[:, j] = np.where(keep, pm[X[:, i]], X[:, j])
    return X


X_train, X_test = sample(30000), sample(5000)
print(X_train.shape, X_test.shape)

## 2. 모델 — 직접 구현한 작은 Transformer 인코더
attention 가중치를 꺼내 볼 수 있도록 `nn.MultiheadAttention`을 직접 사용한다.

In [ ]:
class Block(nn.Module):
    def __init__(self, d, heads):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(d, heads, batch_first=True)
        self.ff = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
        self.last_attn = None

    def forward(self, x):
        h = self.ln1(x)
        a, w = self.attn(h, h, h, need_weights=True, average_attn_weights=False)
        self.last_attn = w.detach()          # (batch, heads, L, L)
        x = x + a
        return x + self.ff(self.ln2(x))


class TinyMLM(nn.Module):
    def __init__(self, d=64, heads=4, layers=2):
        super().__init__()
        self.tok = nn.Embedding(V + 1, d)    # 잔기 20 + 마스크 1
        self.pos = nn.Embedding(L, d)        # 학습되는 위치 임베딩 (고정 길이)
        self.blocks = nn.ModuleList([Block(d, heads) for _ in range(layers)])
        self.ln = nn.LayerNorm(d)
        self.out = nn.Linear(d, V)

    def forward(self, x):
        h = self.tok(x) + self.pos(torch.arange(x.size(1), device=x.device))
        for b in self.blocks:
            h = b(h)
        return self.out(self.ln(h))


model = TinyMLM().to(DEVICE)
print("parameters:", sum(p.numel() for p in model.parameters()))

## 3. MLM 학습
각 위치를 15% 확률로 마스킹하고, 마스킹된 위치의 잔기만 예측한다 (BERT의 80/10/10 규칙은 단순화를 위해 생략).

In [ ]:
MASK_P = 0.15
BATCH = 256
STEPS = 4000
LR = 2e-3

Xtr = torch.from_numpy(X_train).long().to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=STEPS, pct_start=0.1)

losses = []
t0 = time.time()
model.train()
for step in range(STEPS):
    idx = torch.randint(0, Xtr.size(0), (BATCH,), device=DEVICE)
    x = Xtr[idx]
    m = torch.rand(x.shape, device=DEVICE) < MASK_P
    m[:, 0] |= ~m.any(dim=1)                       # 마스크가 하나도 없는 서열 방지
    inp = x.masked_fill(m, MASK_ID)
    logits = model(inp)
    loss = F.cross_entropy(logits[m], x[m])
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
    sched.step()
    losses.append(loss.item())
    if (step + 1) % 500 == 0:
        print(f"step {step + 1:5d}  loss {np.mean(losses[-500:]):.3f}  ({time.time() - t0:.0f}s)")

plt.figure(figsize=(4.5, 3))
plt.plot(np.convolve(losses, np.ones(50) / 50, mode="valid"))
plt.xlabel("step")
plt.ylabel("MLM loss (smoothed)")
plt.tight_layout()
plt.show()

학습 손실은 약 2.0 근처에서 평탄해진다(seed=0, 직접 확인). 0에 가까워지지 않는 이유를 생각해 보자 — 비커플링 위치처럼 문맥으로 줄일 수 없는 불확실성의 하한은 무엇인가?

## 4. 평가 — 한 위치만 가렸을 때의 정확도
테스트 서열에서 위치 p 하나만 마스킹하고 정답 잔기를 맞히는 정확도를 위치별로 잰다.

비교 기준
- **위치별 최빈값 baseline**: 훈련 데이터에서 위치 p에 가장 흔한 잔기를 항상 예측 (문맥을 보지 않음)
- **오라클(상한)**: 커플링 규칙을 아는 경우의 정확도 (규칙으로 파트너로부터 예측)

In [ ]:
@torch.no_grad()
def acc_single_mask(model, X, p, bs=1000):
    model.eval()
    Xt = torch.from_numpy(X).long().to(DEVICE)
    correct = 0
    for s in range(0, Xt.size(0), bs):
        x = Xt[s:s + bs].clone()
        y = x[:, p].clone()
        x[:, p] = MASK_ID
        correct += (model(x)[:, p].argmax(-1) == y).sum().item()
    return correct / Xt.size(0)


model_acc = np.array([acc_single_mask(model, X_test, p) for p in range(L)])

mode_pred = np.array([np.bincount(X_train[:, p], minlength=V).argmax() for p in range(L)])
base_acc = np.array([(X_test[:, p] == mode_pred[p]).mean() for p in range(L)])

oracle = {}
for (i, j), pm in zip(PAIRS, PERMS):
    inv = np.argsort(pm)
    oracle[j] = (pm[X_test[:, i]] == X_test[:, j]).mean()      # i로부터 j 예측
    oracle[i] = (inv[X_test[:, j]] == X_test[:, i]).mean()     # j로부터 i 예측

cp = np.array(coupled_positions)
unc = np.array([p for p in range(L) if p not in set(coupled_positions)])
print(f"coupled positions   : model {model_acc[cp].mean():.3f} | position-mode baseline {base_acc[cp].mean():.3f} | oracle {np.mean([oracle[p] for p in cp]):.3f}")
print(f"uncoupled positions : model {model_acc[unc].mean():.3f} | position-mode baseline {base_acc[unc].mean():.3f}")

colors = ["tab:red" if p in set(coupled_positions) else "tab:gray" for p in range(L)]
fig, ax = plt.subplots(figsize=(9, 3))
ax.bar(range(L), model_acc, color=colors)
ax.plot(range(L), base_acc, "k.", label="position-mode baseline")
ax.set_xlabel("position (red = coupled)")
ax.set_ylabel("accuracy (single mask)")
ax.legend()
plt.tight_layout()
plt.show()

**읽는 법**: 빨간 막대(커플링 위치)가 baseline 점(검은 점)보다 훨씬 높고 회색(비커플링)은 baseline 근처라면,
모델이 **마스킹된 위치의 파트너 잔기를 찾아 활용하는 법**을 MLM 목적함수만으로 학습한 것이다.
비커플링 위치는 문맥에서 얻을 정보가 없으므로 baseline(위치별 최빈값)이 사실상 상한이다.

## 5. attention은 파트너를 보는가?
커플링 위치 j 하나를 마스킹하고, j에서 나가는 attention이 **파트너 i**에 얼마나 집중되는지 본다.
(지표: 파트너에 대한 attention ÷ 나머지 위치 평균 attention, 헤드 평균·샘플 평균)

In [ ]:
@torch.no_grad()
def partner_attention_ratio(model, X, i, j, n=1000):
    model.eval()
    x = torch.from_numpy(X[:n]).long().to(DEVICE).clone()
    x[:, j] = MASK_ID
    model(x)
    ratios = []
    for b in model.blocks:
        a = b.last_attn[:, :, j, :].mean(dim=(0, 1)).cpu().numpy()   # j에서 나가는 attention (L,)
        others = np.delete(a, [i, j])
        ratios.append(a[i] / others.mean())
    return ratios


print("layer1/layer2 에서 '파트너 attention ÷ 나머지 평균':")
all_ratios = np.array([partner_attention_ratio(model, X_test, i, j) for (i, j) in PAIRS])
for k, (i, j) in enumerate(PAIRS):
    print(f"  pair ({i:2d}->{j:2d}): " + "  ".join(f"{r:6.2f}" for r in all_ratios[k]))
print("평균:", np.round(all_ratios.mean(0), 2))

**해석 가이드**: 비율이 1보다 크면 파트너에 평균 이상으로 attention이 모인다는 뜻이다. 층이나 헤드마다 역할이 다를 수 있고(헤드 평균은 희석 효과가 있음),
attention이 곧 설명(explanation)은 아니다. 실제 PLM에서 attention과 contact의 대응은 Rao et al. 2021을 직접 읽어 보자.

## 6. 정리 및 과제
- MLM 목적함수만으로 위치 간 의존성이 학습된다 — 이것이 PLM의 임베딩에 구조·기능 정보가 담기는 직관적 근거다.
- 과제 ([../exercises.md](../exercises.md) 과제 2): `N_PAIRS`, `NOISE`, 훈련 샘플 수, 모델 크기를 바꿔 결과가 어떻게 달라지는지 시드 3회 이상으로 보고하라.
- 주의: 합성 데이터에서 잘 배운다고 해서 실제 단백질 서열의 분포(복잡한 고차 의존성)에서도 같은 학습 효율을 보장하지 않는다.